# 采购增量分析 — 研究验证
口径见 [采购增量分析](采购增量分析.md)。


In [ ]:
import numpy as np
import pandas as pd

from read_data import read_inbound
from analysis.procurement_increment import (
    EXTERNAL_SUPPLIER_TYPES,
    NEW_MATERIAL_LABEL,
    SUPPLIER_TYPE_COLUMN,
    Config,
    export,
    run_analysis,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## 一、取数与基础核对

先确认这份数据本身是干净的：字段类型、日期口径、维度取值。任何一项对不上，后面的指标都不能信。

In [ ]:
# 取数：一个参数读一个完整周期，两个参数读区间，留空读全部归档

START, END = "2026-09", ""
df = read_inbound(START, END)


In [ ]:
def check(name, ok, detail=""):
    return {"核对项": name, "结论": "通过" if ok else "不通过", "说明": detail}


rows = []

# 1. 上游是否已经做过范围过滤
rows.append(check(
    "分析类型只有「入库」",
    SOURCE["分析类型"].nunique() == 1 and SOURCE["分析类型"].iloc[0] == "入库",
    f"取值 {sorted(SOURCE['分析类型'].unique())}",
))
rows.append(check(
    "是否多组织内交易全为 0",
    set(SOURCE["是否多组织内交易"].unique()) == {0},
    f"取值 {sorted(SOURCE['是否多组织内交易'].unique())}",
))

# 2. 板块取值是否与口径一致
boards = sorted(SOURCE["板块"].unique())
rows.append(check("板块只有检验/检验产品/诊断", set(boards) == {"检验", "检验产品", "诊断"}, f"取值 {boards}"))

# 3. 年月列是否与出入库日期一致（决定月维度能不能用）
ym_from_date = pd.to_datetime(SOURCE["出入库日期"], errors="coerce").dt.strftime("%Y-%m")
ym_from_col = SOURCE["年份"].astype(str) + "-" + SOURCE["月份"].astype(str).str.zfill(2)
mismatch = int((ym_from_date != ym_from_col).sum())
rows.append(check("年份月份 = 出入库日期所在月", mismatch == 0, f"不一致 {mismatch} 行"))

# 4. 单价列是否可用（金额×10000/数量 应与 采购不含税单价 一致）
qty = pd.to_numeric(SOURCE["数量"], errors="coerce")
amt = pd.to_numeric(SOURCE["采购不含税金额(万)"], errors="coerce")
unit = pd.to_numeric(SOURCE["采购不含税单价"], errors="coerce")
usable = (qty > 0) & unit.notna()
rel = ((amt * 10000 / qty) - unit).abs() / unit.replace(0, np.nan)
rows.append(check(
    "行级单价 = 金额×10000/数量",
    bool((rel[usable] > 0.01).mean() < 0.001),
    f"相对差 >1% 的行 {int((rel[usable] > 0.01).sum()):,} / {int(usable.sum()):,}",
))

# 5. 物料固定属性是否唯一（决定 groupby().first() 会不会串味）
multi = {c: int((SOURCE.groupby("物料编码")[c].nunique() > 1).sum())
         for c in ("物料名称", "规格", "品牌", "五级分类", "主单位")}
rows.append(check(
    "同一物料编码的固定属性唯一",
    all(v == 0 for k, v in multi.items() if k != "主单位"),
    "；".join(f"{k} {v} 处多值" for k, v in multi.items() if v) or "全部唯一",
))

pd.DataFrame(rows)

### 供应商范围：关联交易必须先剔掉

`财务内部外供应商类型` 里只有「外部单位」是市场交易对象。关联单位之间不是议价关系，
留着会同时污染增长排名和供应商集中度——它们占了采购金额的 25%、数量的 69%。

In [ ]:
type_stats = (
    SOURCE.groupby(SUPPLIER_TYPE_COLUMN, dropna=False)
    .agg(行数=("物料编码", "size"), 物料数=("物料编码", "nunique"),
         数量=("数量", "sum"), 金额万=("采购不含税金额(万)", "sum"))
    .sort_values("金额万", ascending=False)
)
type_stats["金额占比"] = type_stats["金额万"] / type_stats["金额万"].sum()
type_stats["是否保留"] = np.where(
    type_stats.index.isin(EXTERNAL_SUPPLIER_TYPES), "保留", "剔除"
)
type_stats

> 源数据另有一列 `自定义内外部供应商（凯莱谱、生物为外部）`，它给出的内部供应商金额只有 3% 左右，
> 与上表差好几倍。**本分析以 `财务内部外供应商类型` 为准**，两列不能混用。

## 二、口径改动的影响

同一份数据，只切换供应商范围，看候选清单怎么变。这类对比要留痕，否则口径一改就没人说得清数为什么变了。

In [ ]:
WIN = dict(start_date="2025-10-01", end_date="2026-09-30")

compare = {}
for label, external_only in [("仅外部单位（现口径）", True), ("含关联单位（旧口径）", False)]:
    res = run_analysis(Config(external_only=external_only, **WIN))
    full = res["report"].merge(
        res["priority_summary"].set_index("谈判优先级")["物料数"], on="谈判优先级", how="left"
    )
    compare[label] = res["priority_summary"].set_index("谈判优先级")["物料数"]

pd.DataFrame(compare).fillna(0).astype(int)

In [ ]:
# 两种口径下的候选差异明细：只看优先谈判
def priority_set(external_only):
    res = run_analysis(Config(external_only=external_only, **WIN))
    sub = res["report"].query("谈判优先级 == '优先谈判'")
    return res, set(sub["物料编码"])

res_now, set_now = priority_set(True)
res_old, set_old = priority_set(False)

print(f"现口径 优先谈判 {len(set_now)} 个 | 旧口径 {len(set_old)} 个")
print(f"重叠 {len(set_now & set_old)} 个（占现口径 {len(set_now & set_old) / len(set_now):.1%}）")
print(f"仅旧口径有 {len(set_old - set_now)} 个 → 由关联交易带来的候选")
print(f"仅现口径有 {len(set_now - set_old)} 个 → 剔除关联交易后新浮现的候选")

res_now["report"].loc[
    res_now["report"]["物料编码"].isin(set_old - set_now),
    ["物料名称", "最近12个月累计采购不含税金额(万)", "核心供应商名称"],
].head()

## 三、独立复算：月度序列与增长指标

不复用 `build_monthly` / `add_monthly_metrics`，直接用 pandas 从原始明细重算一遍，
再和代码结果逐物料比对。这是发现「代码和文档不一致」最直接的办法。

In [ ]:
analysis_df, meta = __import__("analysis.procurement_increment", fromlist=["clean_and_filter"]).clean_and_filter(
    SOURCE, Config(**WIN)
)
window_start = pd.Timestamp(meta["window_start"] + "-01")
window_end = pd.Timestamp(meta["window_end"] + "-01")

# 独立汇总：物料 × 月
recalc = (
    analysis_df.assign(年月=pd.to_datetime(dict(year=analysis_df["年份"],
                                                month=analysis_df["月份"], day=1)))
    .groupby(["物料编码", "年月"], observed=True)
    .agg(数量=("数量", "sum"), 金额=("采购不含税金额(万)", "sum"))
    .reset_index()
)

# 生产侧的月度序列（已补齐 12 个月、已算指标）
monthly = res_now["monthly"]
merged = monthly.merge(recalc, on=["物料编码", "年月"], suffixes=("_代码", "_复算"), how="outer", indicator=True)

print(f"代码月度行数 {len(monthly):,} | 独立复算行数 {len(recalc):,} | 合并后 {len(merged):,}")
print(f"仅代码有的行 {(merged['_merge'] == 'left_only').sum():,}（应为补齐的零采购月）")
print(f"仅复算有的行 {(merged['_merge'] == 'right_only').sum():,}（应为 0）")
diff_qty = (merged["数量_代码"] - merged["数量_复算"].fillna(0)).abs()
diff_amt = (merged["采购不含税金额(万)"] - merged["金额"].fillna(0)).abs()
print(f"数量最大偏差 {diff_qty.max():,.6f} | 金额最大偏差 {diff_amt.max():,.6f}")

In [ ]:
# 复算趋势指数：最近3个月均值 / 前3个月均值 - 1
pivot = (
    monthly.pivot(index="物料编码", columns="年月", values="数量")
    .sort_index(axis=1)
)
last3 = pivot.iloc[:, -3:].mean(axis=1)
prev3 = pivot.iloc[:, -6:-3].mean(axis=1)
trend_recalc = (last3 / prev3.where(prev3 > 0) - 1).rename("趋势指数_复算")

code_trend = (
    monthly.sort_values("年月").groupby("物料编码").tail(1)
    .set_index("物料编码")["采购量趋势指数"]
)
cmp_trend = pd.concat([code_trend, trend_recalc], axis=1)
cmp_trend["差"] = (cmp_trend["采购量趋势指数"] - cmp_trend["趋势指数_复算"]).abs()

print(f"比对物料 {len(cmp_trend):,} 个")
print(f"两者都为空 {int((cmp_trend['采购量趋势指数'].isna() & cmp_trend['趋势指数_复算'].isna()).sum()):,}")
print(f"最大偏差 {cmp_trend['差'].max():,.10f}")
print(f"是否有 inf：{int(np.isinf(cmp_trend['采购量趋势指数']).sum())} 个（必须为 0）")

## 四、边界与陷阱

四件事容易出错，逐个断言。任何一条不通过，都不应该出报告。

In [ ]:
report = res_now["report"]
checks = []

# 1. 不允许出现 inf（否则 inf 既满足阈值又排在榜首）
checks.append(("趋势指数无 inf", int(np.isinf(report["采购量趋势指数"]).sum()) == 0,
               f"inf {int(np.isinf(report['采购量趋势指数']).sum())} 个"))

# 2. 新引入的必须全部没有趋势指数，非新引入的必须有
new = report["谈判优先级"] == NEW_MATERIAL_LABEL
checks.append(("新引入的趋势指数为空", bool(report.loc[new, "采购量趋势指数"].isna().all()),
               f"新引入 {int(new.sum())} 个，其中为空 {int(report.loc[new, '采购量趋势指数'].isna().sum())} 个"))
checks.append(("非新引入的趋势指数非空", bool(report.loc[~new, "采购量趋势指数"].notna().all()),
               f"缺失 {int(report.loc[~new, '采购量趋势指数'].isna().sum())} 个"))

# 3. 四档互斥且并集就是报告物料（不能有物料落在分类之外）
labels = set(report["谈判优先级"].unique())
expected = {"优先谈判", "观察", "不建议", NEW_MATERIAL_LABEL}
checks.append(("四档标签完整", labels <= expected, f"实际 {sorted(labels)}"))
checks.append(("档位计数合计 = 报告物料数",
               int(res_now["priority_summary"]["物料数"].sum()) == len(report),
               f"{int(res_now['priority_summary']['物料数'].sum())} vs {len(report)}"))

# 4. 报告里的物料必须都过了最近3个月连续采购准入
checks.append(("报告物料最近3个月无零采购",
               bool((report["最近3个月采购量占比"] > 0).all()) if "最近3个月采购量占比" in report else True,
               "占比>0 说明最近3个月都有采购"))

for name, ok, detail in checks:
    print(f"{'✓' if ok else '✗'} {name}：{detail}")
assert all(ok for _, ok, _ in checks), "有核对项不通过，先别出报告"

In [ ]:
# 免费行与退货：解释「数量涨 ≠ 采购额涨」
in_window = SOURCE.loc[
    (pd.to_datetime(SOURCE["出入库日期"]) >= window_start)
    & (pd.to_datetime(SOURCE["出入库日期"]) <= window_end + pd.offsets.MonthEnd(0))
]
free = in_window.loc[in_window["采购不含税金额(万)"] <= 0]
returns = in_window.loc[in_window["出入库类型名称"].isin(["退回供应商", "采购退货"])]

print(f"窗口内明细 {len(in_window):,} 行")
print(f"  有数量无金额（免费/配套）：{len(free):,} 行，占数量 "
      f"{free['数量'].sum() / in_window['数量'].sum():.1%}")
print(f"  退货（数量为负）：{len(returns):,} 行，冲减数量 {returns['数量'].sum():,.0f}，"
      f"冲减金额 {returns['采购不含税金额(万)'].sum():,.1f} 万")

# 优先谈判里免费量占比偏高的物料，谈判空间通常有限
free_share = (
    in_window.groupby("物料编码")
    .apply(lambda d: d.loc[d["采购不含税金额(万)"] <= 0, "数量"].sum() / d["数量"].sum()
           if d["数量"].sum() else np.nan, include_groups=False)
    .rename("免费数量占比")
)
flagged = report.merge(free_share, left_on="物料编码", right_index=True, how="left")
flagged = flagged.loc[(flagged["谈判优先级"] == "优先谈判") & (flagged["免费数量占比"] > 0.5)]
print(f"优先谈判中免费数量占比 >50% 的物料：{len(flagged)} 个")
flagged[["物料名称", "免费数量占比", "最近3个月平均采购量",
         "最近12个月累计采购不含税金额(万)"]].sort_values("免费数量占比", ascending=False).head(10)

## 五、出报告

核对全部通过后，用同一条生产路径导出，确认 notebook 与 `scripts/run_analysis.py` 结论一致。

In [ ]:
final = run_analysis(Config(**WIN))
path = export(final, __import__("paths").OUTPUTS_ROOT)

print(f"报告：{path}")
print(f"窗口：{final['meta']['window_start']} ~ {final['meta']['window_end']}")
print(f"供应商范围：{final['meta']['supplier_scope']}"
      f"（剔除 {final['meta']['supplier_excluded_rows']:,} 行 / "
      f"{final['meta']['supplier_excluded_amount']:,.1f} 万元）")
print(final["priority_summary"].to_string(index=False))

## 六、换数据时要重跑的核对

1. 第一节的基础核对表全部「通过」；
2. 供应商类型分布里是否出现新取值（如新的关联类型），判断是否需要调整 `EXTERNAL_SUPPLIER_TYPES`；
3. 第三、四节的复算偏差为 0、断言全通过；
4. 报告「运行摘要」页的四档物料数与本次对比，避免口径无声漂移。